# Compare two months of campaigns in VS Code

*Python Foundations · Topic 10.3*

The paid media team says that the spring sale started in March 2025, and that every 2024 campaign had stopped by February. Check both claims by comparing the campaigns in February's and March's session exports.

1. In the kernel picker in the top right, choose **Python Environments…** and the `.venv` in `referral-report`. Then click **Run All**. The first cell should show a path in `referral-report`'s `.venv`.
2. In the first cell marked ✏️, create `feb_campaigns` and `march_campaigns`: the sets of `session_campaign` values in `feb_rows` and `march_rows`, with set comprehensions.
3. In the second cell marked ✏️, create `new_campaigns`, the campaigns in March that weren't in February, and `ended_campaigns`, the campaigns in February that weren't in March. End the cell with `sorted(new_campaigns)`.
4. In the last cell marked ✏️, create `running_both`: the campaigns in both months, without the values in `NOT_CAMPAIGNS`. End the cell with `len(running_both)`.
5. Run the **Check your work** cell. Then click **Restart** and **Run All** in the toolbar, check that you get the same result, and save the notebook.
6. Open your `10.2-referral-hostnames.ipynb` in VS Code too, choose the project's `.venv` or the `referral-report` kernel, and run all. Its check should pass again.

**Expected output**: `new_campaigns` has six campaigns, all of them `springsale`, `ended_campaigns` has two 2024 campaigns, and `len(running_both)` is `20`.

In [ ]:
import sys

print(sys.executable)

In [ ]:
import csv

with open("data/sessions_export_2025_02.csv", newline="", encoding="utf-8") as file:
    feb_rows = list(csv.DictReader(file))
with open("data/sessions_export_2025_03.csv", newline="", encoding="utf-8") as file:
    march_rows = list(csv.DictReader(file))

# Values in session_campaign that aren't campaigns
NOT_CAMPAIGNS = {"", "-", "(not set)", "(direct)", "(organic)", "(referral)", "internal_test"}

len(feb_rows), len(march_rows)

In [ ]:
# ✏️ Create feb_campaigns and march_campaigns with set comprehensions


In [ ]:
# ✏️ Create new_campaigns and ended_campaigns, and end the cell with sorted(new_campaigns)


In [ ]:
# ✏️ Create running_both without NOT_CAMPAIGNS, and end the cell with len(running_both)


In [ ]:
# Check your work: run this cell last. If you change anything above, run the cells again from the top first.
# This cell only reads your variables. It doesn't change anything.
def check_your_work():
    import os
    import sys
    from pathlib import Path
    found = globals()
    results = []

    folder = Path.cwd()
    project = next((path for path in [folder, *folder.parents] if (path / "pyproject.toml").exists()), None)
    prefix = Path(sys.prefix).resolve()
    if project is None:
        results.append((False, f"This notebook isn't in a uv project's folder, but in {folder}. Move it into referral-report, with its data folder, and open that folder in VS Code."))
    elif project.name != "referral-report":
        results.append((False, f"This notebook is in the {project.name} project. The exercise uses referral-report, from topic 9.3.: run get_files.py in that folder."))
    elif prefix == (project / ".venv").resolve():
        results.append((True, "The kernel is referral-report's .venv."))
    else:
        results.append((False, f"The kernel is {sys.executable}, not referral-report's .venv. In the kernel picker in the top right, choose Python Environments… and the .venv in referral-report."))

    if "__vsc_ipynb_file__" in found:
        results.append((True, "This notebook runs in VS Code."))
    elif os.environ.get("JPY_SESSION_NAME"):
        results.append((False, "This notebook runs in JupyterLab. This exercise is for VS Code: open the referral-report folder in VS Code, and open the notebook there."))

    expected_feb = {"", "-", "(not set)", "(direct)", "(organic)", "(referral)", "internal_test", "2024_us_email_newsletter", "2024_us_search_generic"}
    expected_feb |= {f"2025_{market}_{name}" for market in ("ca", "us") for name in ("display_prospecting", "display_retargeting", "email_newsletter", "facebook_prospecting", "facebook_retargeting", "instagram_prospecting", "search_brand", "search_generic", "search_shopping", "youtube_awareness")}
    spring = {f"2025_{market}_{channel}_springsale" for market in ("ca", "us") for channel in ("email", "facebook", "search")}
    expected_march = (expected_feb - {"2024_us_email_newsletter", "2024_us_search_generic"}) | spring

    for name, expected, month in [("feb_campaigns", expected_feb, "February"), ("march_campaigns", expected_march, "March")]:
        value = found.get(name)
        if value is None:
            results.append((False, f"{name} isn't defined. Run the cell that creates it."))
        elif isinstance(value, list):
            results.append((False, f"{name} is a list of {len(value)} values. Use curly braces, so it's a set and each campaign appears once."))
        elif value == expected:
            results.append((True, f"{name} has the {len(expected)} values of session_campaign in {month}."))
        elif not isinstance(value, set):
            results.append((False, f"{name} should be a set, but it is a {type(value).__name__}."))
        else:
            results.append((False, f"{name} has {len(value)} values, but {month} has {len(expected)}. Use row[\"session_campaign\"] for every row of {'feb_rows' if month == 'February' else 'march_rows'}."))

    new = found.get("new_campaigns")
    ended = found.get("ended_campaigns")
    ended_expected = {"2024_us_email_newsletter", "2024_us_search_generic"}
    if new is None or ended is None:
        results.append((False, "new_campaigns or ended_campaigns isn't defined. Run the cell that creates them."))
    elif new == spring and ended == ended_expected:
        results.append((True, "new_campaigns has the 6 spring sale campaigns, and ended_campaigns the 2 campaigns that stopped."))
    elif new == ended_expected and ended == spring:
        results.append((False, "new_campaigns and ended_campaigns are the wrong way round. The order matters in a difference: new_campaigns = march_campaigns - feb_campaigns."))
    elif new == expected_feb & expected_march:
        results.append((False, "new_campaigns has the campaigns in both months. Use -, the difference, for the ones in March but not in February."))
    else:
        results.append((False, f"new_campaigns should be {sorted(spring)}, and ended_campaigns {sorted(ended_expected)}."))

    both = found.get("running_both")
    expected_both = (expected_feb & expected_march) - {"", "-", "(not set)", "(direct)", "(organic)", "(referral)", "internal_test"}
    if both is None:
        results.append((False, "running_both isn't defined. Run the cell that creates it."))
    elif both == expected_both:
        results.append((True, "running_both has the 20 campaigns that ran in both months."))
    elif both == expected_feb & expected_march:
        results.append((False, "running_both still has the values that aren't campaigns, such as '(not set)'. Take NOT_CAMPAIGNS away from it with -."))
    elif both == expected_feb | expected_march or both == (expected_feb | expected_march) - {"", "-", "(not set)", "(direct)", "(organic)", "(referral)", "internal_test"}:
        results.append((False, "running_both has the campaigns in either month. Use &, the intersection, for the ones in both."))
    else:
        results.append((False, f"running_both should have 20 campaigns, but it has {len(both) if isinstance(both, set) else both!r}."))

    for passed, message in results:
        print(("✅ " if passed else "❌ ") + message)
    if all(passed for passed, message in results):
        print("\nEverything matches. Now click Restart and Run All, and check that you get the same result.")
    else:
        print("\nFix the first ❌, run the cells above again from the top, then run this cell again.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
feb_campaigns = {row["session_campaign"] for row in feb_rows}
march_campaigns = {row["session_campaign"] for row in march_rows}
```

```python
new_campaigns = march_campaigns - feb_campaigns
ended_campaigns = feb_campaigns - march_campaigns
sorted(new_campaigns)
```

```python
running_both = (feb_campaigns & march_campaigns) - NOT_CAMPAIGNS
len(running_both)
```

```text
uv run https://raw.githubusercontent.com/team-simmer/python-foundations/main/lesson-10/get_files.py 10.3
```

</details>